In [13]:
import json
import os
from dataclasses import dataclass
from typing import List, Dict, Any, Optional
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

RELEVANCE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

NLI_MODEL = "roberta-large-mnli"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [14]:


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

## Relevance Scorer

In [15]:
class RelevanceScorer:
    def __init__(self):
        self.model = SentenceTransformer(RELEVANCE_MODEL, device=DEVICE)

    def score(self, qc_texts, answers, batch_size=64):
        qc_emb = self.model.encode(
            qc_texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=batch_size,
        )
        ans_emb = self.model.encode(
            answers,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=batch_size,
        )
        return (qc_emb * ans_emb).sum(axis=1).tolist()

## Correctness Score NLI

In [16]:
class CorrectnessScorer:
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
        self.model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE)
        self.model.eval()

        id2label = self.model.config.id2label
        self.entailment_idx = [k for k, v in id2label.items() if "entail" in v.lower()][0]

    @torch.inference_mode()
    def score(self, premises, hypotheses, batch_size=16, max_len=384):
        scores = []
        for i in range(0, len(premises), batch_size):
            p = premises[i:i+batch_size]
            h = hypotheses[i:i+batch_size]

            enc = self.tokenizer(
                p, h,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            ).to(DEVICE)

            probs = torch.softmax(self.model(**enc).logits, dim=-1)
            scores.extend(probs[:, self.entailment_idx].cpu().numpy().tolist())
        return scores

## Score Dataset

In [17]:
def process_split(input_json, out_prefix):
    data = load_json(input_json)

    qc_texts = [
        f"Question: {d.get('question','')}\nContext: {d.get('context','')}"
        for d in data
    ]
    answers = [d.get("answer", "") for d in data]

    rel = RelevanceScorer().score(qc_texts, answers)
    corr = CorrectnessScorer().score(qc_texts, answers)

    # concatenate scores back
    for i, row in enumerate(data):
        row["relevance_score"] = float(rel[i])
        row["correctness_score"] = float(corr[i])

    # save JSON
    json_path = f"{out_prefix}_scored.json"
    save_json(json_path, data)

    # save CSV
    df = pd.DataFrame(data)
    csv_path = f"{out_prefix}_scored.csv"
    df.to_csv(csv_path, index=False)

    print(f"Saved: {json_path}")
    print(f"Saved: {csv_path}")
    print(f"Rows: {len(df)}")
    print(
        f"Relevance mean={df.relevance_score.mean():.3f}, "
        f"Correctness mean={df.correctness_score.mean():.3f}"
    )


In [19]:
import pandas as pd
process_split('train.json', 'train')

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Saved: train_scored.json
Saved: train_scored.csv
Rows: 21021
Relevance mean=0.667, Correctness mean=0.656


In [20]:
process_split('test.json', 'test')

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Saved: test_scored.json
Saved: test_scored.csv
Rows: 2000
Relevance mean=0.678, Correctness mean=0.642
